# 🛢️ SIH26143 — U-Net Oil Spill Detection: Training Notebook
**Organization:** National Technical Research Organisation (NTRO)  
**Dataset:** Sentinel-1 SAR Oil Spill Dataset (Zenodo)  
**Model:** U-Net CNN for Binary Segmentation (Water vs Oil Slick)

---
## Pipeline
1. Install dependencies
2. Download dataset from Zenodo
3. Explore & visualise samples
4. Define U-Net model
5. Train with Dice + BCE loss
6. Evaluate on validation set
7. Save best weights for inference

## Step 1 — Install Dependencies

In [1]:
!pip install torch torchvision rasterio opencv-python-headless numpy matplotlib tqdm zenodo-get -q

^C
Traceback (most recent call last):
  File "/Users/ankushkumarguptapahleja/Downloads/sih/.venv/bin/pip", line 6, in <module>
    sys.exit(main())
             ~~~~^^
  File "/Users/ankushkumarguptapahleja/Downloads/sih/.venv/lib/python3.14/site-packages/pip/_internal/cli/main.py", line 86, in main
    return command.main(cmd_args)
           ~~~~~~~~~~~~^^^^^^^^^^
  File "/Users/ankushkumarguptapahleja/Downloads/sih/.venv/lib/python3.14/site-packages/pip/_internal/cli/base_command.py", line 175, in main
    with self.main_context():
         ~~~~~~~~~~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.7/Frameworks/Python.framework/Versions/3.14/lib/python3.14/contextlib.py", line 148, in __exit__
    next(self.gen)
    ~~~~^^^^^^^^^^
  File "/Users/ankushkumarguptapahleja/Downloads/sih/.venv/lib/python3.14/site-packages/pip/_internal/cli/command_context.py", line 20, in main_context
    with self._main_context:
         ^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14

## Step 2 — Download Dataset from Zenodo

Dataset: **Sentinel-1 SAR Oil Spill Dataset**
- Part I → https://zenodo.org/records/8346860
- Part II → https://zenodo.org/records/8253899
- Part III → https://zenodo.org/records/13761290

In [ ]:
import os
import zipfile
import urllib.request
from pathlib import Path

# ── Download configuration ─────────────────────────────────────────
ZENODO_FILES = [
    ("https://zenodo.org/records/8346860/files/Part_1.zip", "Part_1.zip"),
    ("https://zenodo.org/records/8253899/files/Part_2.zip",  "Part_2.zip"),
    # Part III is large (~8 GB); uncomment only if needed
    # ("https://zenodo.org/records/13761290/files/Part_3.zip", "Part_3.zip"),
]

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

def download(url, name):
    dest = RAW_DIR / name
    if dest.exists():
        print(f"✓ Already downloaded: {name}")
        return dest
    print(f"⬇ Downloading {name} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"  Done → {dest}")
    return dest

zips = []
for url, name in ZENODO_FILES:
    zips.append(download(url, name))

# Extract
for zp in zips:
    print(f"📦 Extracting {zp.name} ...")
    with zipfile.ZipFile(zp, 'r') as z:
        z.extractall(RAW_DIR)
print("\n✅ Download complete!")

# List what we have
all_tifs = list(RAW_DIR.rglob("*.tif"))
print(f"Found {len(all_tifs)} .tif files")

⬇ Downloading Part_1.zip ...


HTTPError: HTTP Error 404: NOT FOUND

## Step 3 — Organise into train / val splits

In [ ]:
import shutil
import numpy as np

DATA_DIR = Path("data")

# Collect image-mask pairs
# Zenodo dataset structure: images/ and masks/ subfolders
image_files = sorted(RAW_DIR.rglob("*.tif"))

pairs = []
for img in image_files:
    for mask_suffix in ['masks', 'labels', 'ground_truth']:
        mask_path = img.parent.parent / mask_suffix / img.name
        if mask_path.exists():
            pairs.append((img, mask_path))
            break

print(f"Total image-mask pairs found: {len(pairs)}")

# Train / Val split (80 / 20)
np.random.seed(42)
idxs = np.random.permutation(len(pairs))
n_train = int(len(idxs) * 0.8)
splits = {"train": [pairs[i] for i in idxs[:n_train]],
          "val":   [pairs[i] for i in idxs[n_train:]]}

for subset, file_pairs in splits.items():
    img_dir  = DATA_DIR / subset / "images"
    mask_dir = DATA_DIR / subset / "masks"
    img_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)
    for img_path, mask_path in file_pairs:
        shutil.copy2(img_path,  img_dir  / img_path.name)
        shutil.copy2(mask_path, mask_dir / mask_path.name)
    print(f"  {subset:5s}: {len(file_pairs)} samples")

print("\n✅ Dataset organised!")

## Step 4 — Visualise Sample Images & Masks

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def load_tif(path):
    with rasterio.open(path) as src:
        img = src.read(1).astype(np.float32)
    img = np.clip(img, -30, 10)
    img = (img + 30) / 40.0
    return np.nan_to_num(img)

sample_imgs  = sorted((DATA_DIR / "train" / "images").glob("*.tif"))[:4]
sample_masks = [DATA_DIR / "train" / "masks" / p.name for p in sample_imgs]

fig, axes = plt.subplots(4, 2, figsize=(10, 16))
fig.suptitle("Sample SAR Images & Oil-Spill Masks", fontsize=14, fontweight='bold')

for i, (ip, mp) in enumerate(zip(sample_imgs, sample_masks)):
    img  = load_tif(ip)
    with rasterio.open(mp) as src:
        mask = src.read(1)

    axes[i][0].imshow(img,  cmap='gray')
    axes[i][0].set_title(f"SAR Image: {ip.name[:30]}")
    axes[i][0].axis('off')

    axes[i][1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
    axes[i][1].set_title(f"Mask (red = oil slick)")
    axes[i][1].axis('off')

    spill_pct = (mask > 0).sum() / mask.size * 100
    print(f"  {ip.name[:40]:40s}  oil coverage: {spill_pct:.2f}%")

plt.tight_layout()
plt.savefig("sample_visualisation.png", dpi=120)
plt.show()
print("\nSaved → sample_visualisation.png")

## Step 5 — Define U-Net Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x): return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch//2, 2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        dy = x2.size(2) - x1.size(2); dx = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [dx//2, dx-dx//2, dy//2, dy-dy//2])
        return self.conv(torch.cat([x2, x1], dim=1))

class UNet(nn.Module):
    """U-Net for SAR Oil-Spill Segmentation (1-channel input, binary output)"""
    def __init__(self, n_channels=1, n_classes=1):
        super().__init__()
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        self.up1   = Up(1024, 512)
        self.up2   = Up(512, 256)
        self.up3   = Up(256, 128)
        self.up4   = Up(128, 64)
        self.outc  = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x  = self.up1(x5, x4)
        x  = self.up2(x, x3)
        x  = self.up3(x, x2)
        x  = self.up4(x, x1)
        return self.outc(x)

# Quick architecture test
model = UNet(n_channels=1, n_classes=1)
test  = torch.randn(1, 1, 256, 256)
out   = model(test)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ U-Net ready!")
print(f"   Input  : {list(test.shape)}")
print(f"   Output : {list(out.shape)}")
print(f"   Params : {total_params/1e6:.2f}M")

## Step 6 — Dataset Class & DataLoaders

In [ ]:
import cv2
from torch.utils.data import Dataset, DataLoader

class OilSpillDataset(Dataset):
    def __init__(self, img_dir, mask_dir, size=256, augment=True):
        self.imgs    = sorted(Path(img_dir).glob("*.tif"))
        self.mdir    = Path(mask_dir)
        self.size    = size
        self.augment = augment
        assert len(self.imgs) > 0, f"No images in {img_dir}"
        print(f"  {('train' if augment else 'val'):5s}: {len(self.imgs)} samples")

    def __len__(self): return len(self.imgs)

    def _load_img(self, p):
        with rasterio.open(p) as s: img = s.read(1).astype(np.float32)
        img = np.clip(img, -30, 10); img = (img + 30) / 40.0
        return np.nan_to_num(img)

    def _load_mask(self, p):
        if p.suffix in ('.tif', '.tiff'):
            with rasterio.open(p) as s: m = s.read(1).astype(np.float32)
        else:
            m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(np.float32)
            m = (m > 127).astype(np.float32)
        return m

    def __getitem__(self, idx):
        ip = self.imgs[idx]
        mp = self.mdir / ip.name
        img  = cv2.resize(self._load_img(ip),  (self.size, self.size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(self._load_mask(mp), (self.size, self.size), interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand() > 0.5: img, mask = np.fliplr(img), np.fliplr(mask)
            if np.random.rand() > 0.5: img, mask = np.flipud(img), np.flipud(mask)
            k = np.random.choice([0,1,2,3])
            img, mask = np.rot90(img,k).copy(), np.rot90(mask,k).copy()
        return torch.from_numpy(img).unsqueeze(0), torch.from_numpy(mask).unsqueeze(0)

BATCH = 8
train_ds = OilSpillDataset("data/train/images", "data/train/masks", augment=True)
val_ds   = OilSpillDataset("data/val/images",   "data/val/masks",   augment=False)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f"\n  Train batches: {len(train_dl)} | Val batches: {len(val_dl)}")

## Step 7 — Loss Functions & Metrics

In [ ]:
class DiceLoss(nn.Module):
    """Handles class imbalance (oil << water)"""
    def __init__(self, smooth=1.):
        super().__init__(); self.smooth = smooth
    def forward(self, pred, tgt):
        pred = torch.sigmoid(pred).view(-1); tgt = tgt.view(-1)
        i = (pred * tgt).sum()
        return 1 - (2*i + self.smooth) / (pred.sum() + tgt.sum() + self.smooth)

class CombinedLoss(nn.Module):
    """BCE (with class weight) + Dice"""
    def __init__(self):
        super().__init__()
        self.bce  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]))  # oil pixels are rare
        self.dice = DiceLoss()
    def forward(self, pred, tgt):
        return 0.5 * self.bce(pred, tgt) + 0.5 * self.dice(pred, tgt)

def iou(pred, tgt, t=0.5):
    p = (torch.sigmoid(pred) > t).float()
    i = (p * tgt).sum(); u = p.sum() + tgt.sum() - i
    return (i / (u + 1e-6)).item()

def dice(pred, tgt, t=0.5):
    p = (torch.sigmoid(pred) > t).float()
    i = (p * tgt).sum()
    return (2*i / (p.sum() + tgt.sum() + 1e-6)).item()

print("✅ Loss functions ready")
print("   BCE + Dice (50/50) with pos_weight=10 for class imbalance")

## Step 8 — Train the U-Net

In [ ]:
import json
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ── Config ──────────────────────────────────────────────────────────
EPOCHS   = 50
LR       = 1e-4
SAVE_DIR = Path("checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

device    = torch.device('cuda' if torch.cuda.is_available() else 
                         'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

model     = UNet(n_channels=1, n_classes=1).to(device)
criterion = CombinedLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5, verbose=True)

best_iou = 0.0
history  = []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    tr_loss = tr_iou = tr_dice = 0.0
    pbar = tqdm(train_dl, desc=f"Epoch {epoch:>3}/{EPOCHS} [Train]", leave=False)
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = criterion(preds, masks)
        loss.backward(); optimizer.step()
        tr_loss += loss.item(); tr_iou += iou(preds, masks); tr_dice += dice(preds, masks)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    # ── Validate ──
    model.eval()
    va_loss = va_iou = va_dice = 0.0
    with torch.no_grad():
        for imgs, masks in val_dl:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            va_loss += criterion(preds, masks).item()
            va_iou  += iou(preds, masks); va_dice += dice(preds, masks)

    n_tr, n_va = len(train_dl), len(val_dl)
    tr_loss /= n_tr; tr_iou /= n_tr; tr_dice /= n_tr
    va_loss /= n_va; va_iou /= n_va; va_dice /= n_va
    scheduler.step(va_iou)

    is_best = va_iou > best_iou
    if is_best:
        best_iou = va_iou
        torch.save(model.state_dict(), SAVE_DIR / "best_model.pt")

    # Latest checkpoint
    torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(), 'best_iou': best_iou},
               SAVE_DIR / "latest.pt")

    row = dict(epoch=epoch, tr_loss=round(tr_loss,4), tr_iou=round(tr_iou,4), tr_dice=round(tr_dice,4),
               va_loss=round(va_loss,4), va_iou=round(va_iou,4), va_dice=round(va_dice,4))
    history.append(row)

    star = " ⭐" if is_best else ""
    print(f"Epoch {epoch:>3}/{EPOCHS} │ "
          f"Train Loss {tr_loss:.4f}  IoU {tr_iou:.4f}  Dice {tr_dice:.4f} │ "
          f"Val Loss {va_loss:.4f}  IoU {va_iou:.4f}  Dice {va_dice:.4f}{star}")

with open(SAVE_DIR / "history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"\n✅ Training complete! Best Val IoU: {best_iou:.4f}")
print(f"   Best weights: checkpoints/best_model.pt")

## Step 9 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_x  = [r['epoch']   for r in history]
tr_ious   = [r['tr_iou']  for r in history]
va_ious   = [r['va_iou']  for r in history]
tr_losses = [r['tr_loss'] for r in history]
va_losses = [r['va_loss'] for r in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("U-Net Training — SIH26143 Oil Spill Detection", fontsize=13, fontweight='bold')

ax1.plot(epochs_x, tr_losses, label='Train Loss', color='#ef4444', linewidth=2)
ax1.plot(epochs_x, va_losses, label='Val Loss',   color='#f97316', linestyle='--', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss (BCE + Dice)')
ax1.set_title('Loss Curve'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_x, tr_ious, label='Train IoU', color='#3b82f6', linewidth=2)
ax2.plot(epochs_x, va_ious, label='Val IoU',   color='#22c55e', linestyle='--', linewidth=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('IoU Score')
ax2.set_title('IoU Score Curve'); ax2.legend(); ax2.grid(alpha=0.3)
ax2.axhline(best_iou, color='gold', linestyle=':', label=f'Best IoU: {best_iou:.4f}')
ax2.legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()
print("Saved → training_curves.png")

## Step 10 — Inference on Validation Samples

In [ ]:
# Load best model
model.load_state_dict(torch.load(SAVE_DIR / "best_model.pt", map_location=device))
model.eval()

# Get a batch from val
imgs, masks = next(iter(val_dl))
imgs = imgs.to(device)

with torch.no_grad():
    preds = torch.sigmoid(model(imgs))

# Visualise first 4
fig, axes = plt.subplots(4, 3, figsize=(12, 16))
fig.suptitle("Inference Results: SAR Image | Ground Truth | Prediction", fontsize=12, fontweight='bold')

for i in range(min(4, len(imgs))):
    img_np  = imgs[i,0].cpu().numpy()
    mask_np = masks[i,0].numpy()
    pred_np = (preds[i,0].cpu().numpy() > 0.5).astype(float)

    axes[i][0].imshow(img_np,  cmap='gray');        axes[i][0].set_title("SAR Image");      axes[i][0].axis('off')
    axes[i][1].imshow(mask_np, cmap='Reds', vmin=0, vmax=1); axes[i][1].set_title("Ground Truth"); axes[i][1].axis('off')
    axes[i][2].imshow(pred_np, cmap='Reds', vmin=0, vmax=1); axes[i][2].set_title("U-Net Prediction"); axes[i][2].axis('off')

    sample_iou = iou(model(imgs[i:i+1]), masks[i:i+1].to(device))
    axes[i][2].set_title(f"Prediction  (IoU: {sample_iou:.3f})")

plt.tight_layout()
plt.savefig("inference_results.png", dpi=150)
plt.show()
print("\n✅ Saved → inference_results.png")
print(f"   Best model weights : checkpoints/best_model.pt")
print(f"   Use this in inference.py with SlickDetector(model_path='checkpoints/best_model.pt')")

NameError: name 'model' is not defined